In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
branch_table = dbutils.widgets.get("branch_table")
office_table = dbutils.widgets.get("office_table")
clientepisodefsall_table = dbutils.widgets.get("clientepisodefsall_table")
clientepisodesall_table = dbutils.widgets.get("clientepisodesall_table")
aradjustments_table = dbutils.widgets.get("aradjustments_table")
acthistory_table = dbutils.widgets.get("acthistory_table")
payortype_table = dbutils.widgets.get("payortype_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW demographics_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS INT) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  CAST(MedRecNbr AS STRING) AS MedRecNbr,
  NULL AS PhysicianName,
  NULL AS PhysicianNPI,
  CAST(PhysicianID AS STRING) AS PhysicianID,
  NULL AS AdmitPhysicianName,
  NULL AS AdmitPhysicianNPI,
  NULL AS AdmitPhysicianID,
  NULL AS ReferPhysicianName,
  NULL AS ReferPhysicianNPI,
  CAST(ReferPhysicianID AS STRING) AS ReferPhysicianID,
  NULL AS PrimarySurgeonName,
  NULL AS PrimarySurgeonNPI,
  NULL AS PrimarySurgeonID,
  CAST(AdmitDate AS DATE) AS AdmitDate,
  CAST(DischargeDate AS DATE) AS DischargeDate,
  NULL AS Registrar,
  CAST(ARStatus AS STRING) AS ARStatus,
  CAST(PatType AS STRING) AS PatType,
  CAST(Location AS STRING) AS Location,
  NULL AS DRGCode,
  NULL AS DRGDesc,
  NULL AS DRGWeight,
  NULL AS DRGVersion,
  CAST(AdmitFC AS STRING) AS AdmitFC,
  CAST(AdmitFinClass AS STRING) AS AdmitFinClass,
  CAST(CurrentFC AS STRING) AS CurrentFC,
  CAST(CurrentFinClass AS STRING) AS CurrentFinClass,
  CAST(TotalCharges AS DOUBLE) AS TotalCharges,
  CAST(TotalPymt AS DOUBLE) AS TotalPymt,
  NULL AS TotalInsPymt,
  NULL AS TotalPatientPymt,
  NULL AS TotalAdj,
  NULL AS TotalInsAdj,
  NULL AS TotalPatientAdj,
  CAST(InsBalance AS DOUBLE) AS InsBalance,
  NULL AS PatBalance,
  NULL AS ExpNetRev,
  CAST(PatFName AS STRING) AS PatFName,
  CAST(PatMName AS STRING) AS PatMName,
  CAST(PatLName AS STRING) AS PatLName,
  NULL AS PatSuffix,
  CAST(PatDOB AS DATE) AS PatDOB,
  NULL AS PatSSN,
  CAST(PatGender AS STRING) AS PatGender,
  CAST(PatAddr1 AS STRING) AS PatAddr1,
  NULL AS PatAddr2,
  CAST(PatCity AS STRING) AS PatCity,
  CAST(PatState AS STRING) AS PatState,
  CAST(PatZip AS STRING) AS PatZip,
  NULL AS PatCountry,
  NULL AS PatProvince,
  CAST(PatHomePhone AS STRING) AS PatHomePhone,
  NULL AS PatWorkPhone,
  NULL AS PatCellPhone,
  NULL AS PatEmployer,
  NULL AS GuarRelationship,
  NULL AS Guarantor,
  NULL AS GuarFName,
  NULL AS GuarMName,
  NULL AS GuarLName,
  NULL AS GuarSuffix,
  NULL AS GuarDOB,
  NULL AS GuarSSN,
  NULL AS GuarGender,
  NULL AS GuarAddr1,
  NULL AS GuarAddr2,
  NULL AS GuarCity,
  NULL AS GuarState,
  NULL AS GuarZip,
  NULL AS GuarCountry,
  NULL AS GuarProvince,
  NULL AS GuarHomePhone,
  NULL AS GuarWorkPhone,
  NULL AS GuarCellPhone,
  NULL AS GuarEmployer,
  CAST(LastBillDate AS INT) AS LastBillDate,
  NULL AS LastBillSubmitDate,
  NULL AS LastBillType,
  NULL AS LastBillMediaType,
  CAST(Agency AS STRING) AS Agency,
  NULL AS AgencyAssignDate,
  NULL AS AgencyReturnDate,
  NULL AS AgencyReturnReason,
  NULL AS BadDebtDate,
  NULL AS BadDebtAmt,
  NULL AS BadDebtBal,
  NULL AS ActiveCOB,
  CAST(PatientFacilityID AS STRING) AS PatientFacilityID,
  NULL AS PatientFacilityName,
  NULL AS PatientFacilityNPI,
  NULL AS PatientFacilityType,
  NULL AS CBSA,
  NULL AS LocationCode,
  NULL AS PatientStatusatBill,
  NULL AS PatientStatusatCodeBill,
  NULL AS LastClaimComment,
  CAST(StatementFromDate AS INT) AS StatementFromDate,
  CAST(StatementThroughDate AS INT) AS StatementThroughDate,
  CAST(DateOfServiceFromDate AS INT) AS DateOfServiceFromDate,
  CAST(DateOfServiceThroughDate AS INT) AS DateOfServiceThroughDate,
  NULL AS MedicalDirectorName,
  NULL AS MedicalDirectorNPI,
  CAST(MedicalDirectorID AS STRING) AS MedicalDirectorID,
  CAST(BenefitPeriodID AS INT) AS BenefitPeriodID,
  CAST(BenefitPeriodStartDate AS INT) AS BenefitPeriodStartDate,
  CAST(BenefitPeriodEndDate AS INT) AS BenefitPeriodEndDate,
  NULL AS BenefitPeriodStatus,
  NULL AS EpisodeID,
  CAST(StartofEpisode AS INT) AS StartofEpisode,
  CAST(EndofEpisode AS INT) AS EndofEpisode,
  CAST(StartofPeriod AS INT) AS StartofPeriod,
  CAST(EndofPeriod AS INT) AS EndofPeriod,
  CAST(EpisodeStatus AS STRING) AS EpisodeStatus,
  CAST(ClaimFromDate AS INT) AS ClaimFromDate,
  CAST(ClaimThroughDate AS INT) AS ClaimThroughDate,
  CAST(SourceSystemKey AS INT) AS SourceSystemKey,
  CAST(AcctBalance AS DOUBLE) AS AcctBalance
FROM (
  WITH 
  demographics_cte AS (
    SELECT
      CAST('{fetch_date}' AS DATE) AS ReportingDate,
      CASE  
          WHEN b.branch_code RLIKE '[A-Za-z]' THEN ofc.OfficeNumber  
          ELSE b.branch_code  
      END AS FacilityCode, 
      bi_h.i_id AS AcctNbr, 
      epi.epi_mrnum AS MedRecNbr,
      epi.epi_phid1 AS PhysicianID,
      epi.epi_phid1 AS ReferPhysicianID,
      date_format(epi.epi_StartOfEpisode, 'yyyy-MM-dd') AS AdmitDate,
      date_format(epi.epi_DischargeDate, 'yyyy-MM-dd') AS DischargeDate,
      CASE WHEN bi_h.i_balance > 0 THEN 'Open' ELSE 'Closed' END AS ARStatus,
      'Home Health' AS PatType,
      c.State AS Location,
      pt.PayorType AS AdmitFC,
      pt.PayorType AS AdmitFinClass,
      bi_h.i_charge AS CurrentFC,
      bi_h.i_balance AS CurrentFinClass,
      -- bi_h.i_charge AS TotalCharges,
      NULL AS TotalCharges,
      -- pymt.TotalPayments AS TotalPymt,
      NULL AS TotalPymt,
      bi_h.i_charge AS InsBalance,
      c.FirstName AS PatFName,
      c.MI AS PatMName,
      c.LastName AS PatLName,
      date_format(c.DateOfBirth, 'yyyy-MM-dd') AS PatDOB,
      CASE 
          WHEN c.Gender IN ('Male', 'Female', 'MALE', 'FEMALE') 
          THEN SUBSTRING(c.Gender, 1, 1) 
          ELSE NULL 
      END AS PatGender,
      c.Address AS PatAddr1,
      c.City AS PatCity,
      c.State AS PatState,
      REGEXP_REPLACE(c.ZipCode, '-$', '') AS PatZip,
      CONCAT(c.AreaCode, REGEXP_REPLACE(c.PhoneNumber, '[^0-9]', '')) AS PatHomePhone,
      date_format(bi_h.i_postdate, 'yyyyMMdd') AS LastBillDate,
      '1' AS Agency,
      epi.epi_id AS PatientFacilityID,
      date_format(epi.epi_StartOfEpisode, 'yyyyMMdd') AS StatementFromDate,
      NULL AS StatementThroughDate,  -- No source data available --missing
      date_format(epi.epi_StartOfEpisode, 'yyyyMMdd') AS DateOfServiceFromDate,
      NULL AS DateOfServiceThroughDate,  -- No source data available --missing
      epi.epi_id AS MedicalDirectorID,
      date_format(epi.epi_StartOfEpisode, 'yyyyMMdd') AS BenefitPeriodID,
      date_format(epi.epi_StartOfEpisode, 'yyyyMMdd') AS BenefitPeriodStartDate,
      date_format(epi.epi_DischargeDate, 'yyyyMMdd') AS BenefitPeriodEndDate,
      date_format(epi.epi_StartOfEpisode, 'yyyyMMdd') AS StartofEpisode,
      date_format(epi.epi_DischargeDate, 'yyyyMMdd') AS EndofEpisode,
      date_format(epi.epi_StartOfEpisode, 'yyyyMMdd') AS StartofPeriod,
      NULL AS EndofPeriod,  -- No source data available --missing
      NULL AS EpisodeStatus,
      date_format(epi.epi_StartOfEpisode, 'yyyyMMdd') AS ClaimFromDate,
      NULL AS ClaimThroughDate,  -- No source data available --missing
      '6' AS SourceSystemKey,
      bi_h.i_charge AS AcctBalance
    FROM {source_table} bi_h 
    JOIN {branch_table} b  
        ON bi_h.i_branchcode = b.branch_code 
    LEFT JOIN {office_table} ofc 
        ON ofc.OfficeAbbreviation = b.branch_code
    JOIN {clientepisodefsall_table} cefs
        ON cefs.cefs_id = bi_h.i_cefsid
    JOIN {clientepisodesall_table} epi
        ON epi.epi_id = cefs.cefs_epiid
    LEFT JOIN {payortype_table} pt
        ON pt.pt_id = cefs.cefs_ptid
    JOIN (
      SELECT *,
          ROW_NUMBER() OVER (PARTITION BY pa_id ORDER BY ClientID DESC) as rn
      FROM prd_bronze_raw.hchb_dw_mart.client
      WHERE Deleted = false
    ) c
      ON c.pa_id = epi.epi_paid
      AND c.rn = 1
  -- LEFT JOIN (
  --     SELECT 
  --         ct_iid,
  --         SUM(ct_initialamount) AS TotalPayments
  --     FROM {acthistory_table}
  --     WHERE ct_cttid = 2
  --     GROUP BY ct_iid
  -- ) pymt
  --     ON pymt.ct_iid = bi_h.i_id
  -- LEFT JOIN (
  --     SELECT 
  --         invnum,
  --         SUM(adj_amt) AS TotalAdj
  --     FROM {aradjustments_table}
  --     GROUP BY invnum
  -- ) adj
  --     ON adj.invnum = bi_h.i_id
    WHERE bi_h.i_Balance <> 0
  ),
  demographics_clean AS (
    SELECT *,
    row_number() OVER (PARTITION BY AcctNbr ORDER BY AcctNbr ) AS rn
    FROM demographics_cte
  )
SELECT 
  ReportingDate,
  FacilityCode,
  AcctNbr,
  MedRecNbr,
  PhysicianID,
  ReferPhysicianID,
  AdmitDate,
  DischargeDate,
  ARStatus,
  PatType,
  Location,
  AdmitFC,
  AdmitFinClass,
  CurrentFC,
  CurrentFinClass,
  TotalCharges,
  TotalPymt,
  InsBalance,
  PatFName,
  PatMName,
  PatLName,
  PatDOB,
  PatGender,
  PatAddr1,
  PatCity,
  PatState,
  PatZip,
  PatHomePhone,
  LastBillDate,
  Agency,
  PatientFacilityID,
  StatementFromDate,
  StatementThroughDate,
  DateOfServiceFromDate,
  DateOfServiceThroughDate,
  MedicalDirectorID,
  BenefitPeriodID,
  BenefitPeriodStartDate,
  BenefitPeriodEndDate,
  StartofEpisode,
  EndofEpisode,
  StartofPeriod,
  EndofPeriod,
  EpisodeStatus,
  ClaimFromDate,
  ClaimThroughDate,
  SourceSystemKey,
  AcctBalance
FROM demographics_clean
WHERE rn=1
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING demographics_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 6

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.MedRecNbr = src.MedRecNbr,
    tgt.PhysicianName = src.PhysicianName,
    tgt.PhysicianNPI = src.PhysicianNPI,
    tgt.PhysicianID = src.PhysicianID,
    tgt.AdmitPhysicianName = src.AdmitPhysicianName,
    tgt.AdmitPhysicianNPI = src.AdmitPhysicianNPI,
    tgt.AdmitPhysicianID = src.AdmitPhysicianID,
    tgt.ReferPhysicianName = src.ReferPhysicianName,
    tgt.ReferPhysicianNPI = src.ReferPhysicianNPI,
    tgt.ReferPhysicianID = src.ReferPhysicianID,
    tgt.PrimarySurgeonName = src.PrimarySurgeonName,
    tgt.PrimarySurgeonNPI = src.PrimarySurgeonNPI,
    tgt.PrimarySurgeonID = src.PrimarySurgeonID,
    tgt.AdmitDate = src.AdmitDate,
    tgt.DischargeDate = src.DischargeDate,
    tgt.Registrar = src.Registrar,
    tgt.ARStatus = src.ARStatus,
    tgt.PatType = src.PatType,
    tgt.Location = src.Location,
    tgt.DRGCode = src.DRGCode,
    tgt.DRGDesc = src.DRGDesc,
    tgt.DRGWeight = src.DRGWeight,
    tgt.DRGVersion = src.DRGVersion,
    tgt.AdmitFC = src.AdmitFC,
    tgt.AdmitFinClass = src.AdmitFinClass,
    tgt.CurrentFC = src.CurrentFC,
    tgt.CurrentFinClass = src.CurrentFinClass,
    tgt.TotalCharges = src.TotalCharges,
    tgt.TotalPymt = src.TotalPymt,
    tgt.TotalInsPymt = src.TotalInsPymt,
    tgt.TotalPatientPymt = src.TotalPatientPymt,
    tgt.TotalAdj = src.TotalAdj,
    tgt.TotalInsAdj = src.TotalInsAdj,
    tgt.TotalPatientAdj = src.TotalPatientAdj,
    tgt.AcctBalance = src.AcctBalance,
    tgt.InsBalance = src.InsBalance,
    tgt.PatBalance = src.PatBalance,
    tgt.ExpNetRev = src.ExpNetRev,
    tgt.PatFName = src.PatFName,
    tgt.PatMName = src.PatMName,
    tgt.PatLName = src.PatLName,
    tgt.PatSuffix = src.PatSuffix,
    tgt.PatDOB = src.PatDOB,
    tgt.PatSSN = src.PatSSN,
    tgt.PatGender = src.PatGender,
    tgt.PatAddr1 = src.PatAddr1,
    tgt.PatAddr2 = src.PatAddr2,
    tgt.PatCity = src.PatCity,
    tgt.PatState = src.PatState,
    tgt.PatZip = src.PatZip,
    tgt.PatCountry = src.PatCountry,
    tgt.PatProvince = src.PatProvince,
    tgt.PatHomePhone = src.PatHomePhone,
    tgt.PatWorkPhone = src.PatWorkPhone,
    tgt.PatCellPhone = src.PatCellPhone,
    tgt.PatEmployer = src.PatEmployer,
    tgt.GuarRelationship = src.GuarRelationship,
    tgt.Guarantor = src.Guarantor,
    tgt.GuarFName = src.GuarFName,
    tgt.GuarMName = src.GuarMName,
    tgt.GuarLName = src.GuarLName,
    tgt.GuarSuffix = src.GuarSuffix,
    tgt.GuarDOB = src.GuarDOB,
    tgt.GuarSSN = src.GuarSSN,
    tgt.GuarGender = src.GuarGender,
    tgt.GuarAddr1 = src.GuarAddr1,
    tgt.GuarAddr2 = src.GuarAddr2,
    tgt.GuarCity = src.GuarCity,
    tgt.GuarState = src.GuarState,
    tgt.GuarZip = src.GuarZip,
    tgt.GuarCountry = src.GuarCountry,
    tgt.GuarProvince = src.GuarProvince,
    tgt.GuarHomePhone = src.GuarHomePhone,
    tgt.GuarWorkPhone = src.GuarWorkPhone,
    tgt.GuarCellPhone = src.GuarCellPhone,
    tgt.GuarEmployer = src.GuarEmployer,
    tgt.LastBillDate = src.LastBillDate,
    tgt.LastBillSubmitDate = src.LastBillSubmitDate,
    tgt.LastBillType = src.LastBillType,
    tgt.LastBillMediaType = src.LastBillMediaType,
    tgt.Agency = src.Agency,
    tgt.AgencyAssignDate = src.AgencyAssignDate,
    tgt.AgencyReturnDate = src.AgencyReturnDate,
    tgt.AgencyReturnReason = src.AgencyReturnReason,
    tgt.BadDebtDate = src.BadDebtDate,
    tgt.BadDebtAmt = src.BadDebtAmt,
    tgt.BadDebtBal = src.BadDebtBal,
    tgt.ActiveCOB = src.ActiveCOB,
    tgt.PatientFacilityID = src.PatientFacilityID,
    tgt.PatientFacilityName = src.PatientFacilityName,
    tgt.PatientFacilityNPI = src.PatientFacilityNPI,
    tgt.PatientFacilityType = src.PatientFacilityType,
    tgt.CBSA = src.CBSA,
    tgt.LocationCode = src.LocationCode,
    tgt.PatientStatusatBill = src.PatientStatusatBill,
    tgt.PatientStatusatCodeBill = src.PatientStatusatCodeBill,
    tgt.LastClaimComment = src.LastClaimComment,
    tgt.StatementFromDate = src.StatementFromDate,
    tgt.StatementThroughDate = src.StatementThroughDate,
    tgt.DateOfServiceFromDate = src.DateOfServiceFromDate,
    tgt.DateOfServiceThroughDate = src.DateOfServiceThroughDate,
    tgt.MedicalDirectorName = src.MedicalDirectorName,
    tgt.MedicalDirectorNPI = src.MedicalDirectorNPI,
    tgt.MedicalDirectorID = src.MedicalDirectorID,
    tgt.BenefitPeriodID = src.BenefitPeriodID,
    tgt.BenefitPeriodStartDate = src.BenefitPeriodStartDate,
    tgt.BenefitPeriodEndDate = src.BenefitPeriodEndDate,
    tgt.BenefitPeriodStatus = src.BenefitPeriodStatus,
    tgt.EpisodeID = src.EpisodeID,
    tgt.StartofEpisode = src.StartofEpisode,
    tgt.EndofEpisode = src.EndofEpisode,
    tgt.StartofPeriod = src.StartofPeriod,
    tgt.EndofPeriod = src.EndofPeriod,
    tgt.EpisodeStatus = src.EpisodeStatus,
    tgt.ClaimFromDate = src.ClaimFromDate,
    tgt.ClaimThroughDate = src.ClaimThroughDate,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    MedRecNbr,
    PhysicianName,
    PhysicianNPI,
    PhysicianID,
    AdmitPhysicianName,
    AdmitPhysicianNPI,
    AdmitPhysicianID,
    ReferPhysicianName,
    ReferPhysicianNPI,
    ReferPhysicianID,
    PrimarySurgeonName,
    PrimarySurgeonNPI,
    PrimarySurgeonID,
    AdmitDate,
    DischargeDate,
    Registrar,
    ARStatus,
    PatType,
    Location,
    DRGCode,
    DRGDesc,
    DRGWeight,
    DRGVersion,
    AdmitFC,
    AdmitFinClass,
    CurrentFC,
    CurrentFinClass,
    TotalCharges,
    TotalPymt,
    TotalInsPymt,
    TotalPatientPymt,
    TotalAdj,
    TotalInsAdj,
    TotalPatientAdj,
    AcctBalance,
    InsBalance,
    PatBalance,
    ExpNetRev,
    PatFName,
    PatMName,
    PatLName,
    PatSuffix,
    PatDOB,
    PatSSN,
    PatGender,
    PatAddr1,
    PatAddr2,
    PatCity,
    PatState,
    PatZip,
    PatCountry,
    PatProvince,
    PatHomePhone,
    PatWorkPhone,
    PatCellPhone,
    PatEmployer,
    GuarRelationship,
    Guarantor,
    GuarFName,
    GuarMName,
    GuarLName,
    GuarSuffix,
    GuarDOB,
    GuarSSN,
    GuarGender,
    GuarAddr1,
    GuarAddr2,
    GuarCity,
    GuarState,
    GuarZip,
    GuarCountry,
    GuarProvince,
    GuarHomePhone,
    GuarWorkPhone,
    GuarCellPhone,
    GuarEmployer,
    LastBillDate,
    LastBillSubmitDate,
    LastBillType,
    LastBillMediaType,
    Agency,
    AgencyAssignDate,
    AgencyReturnDate,
    AgencyReturnReason,
    BadDebtDate,
    BadDebtAmt,
    BadDebtBal,
    ActiveCOB,
    PatientFacilityID,
    PatientFacilityName,
    PatientFacilityNPI,
    PatientFacilityType,
    CBSA,
    LocationCode,
    PatientStatusatBill,
    PatientStatusatCodeBill,
    LastClaimComment,
    StatementFromDate,
    StatementThroughDate,
    DateOfServiceFromDate,
    DateOfServiceThroughDate,
    MedicalDirectorName,
    MedicalDirectorNPI,
    MedicalDirectorID,
    BenefitPeriodID,
    BenefitPeriodStartDate,
    BenefitPeriodEndDate,
    BenefitPeriodStatus,
    EpisodeID,
    StartofEpisode,
    EndofEpisode,
    StartofPeriod,
    EndofPeriod,
    EpisodeStatus,
    ClaimFromDate,
    ClaimThroughDate,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.MedRecNbr,
    src.PhysicianName,
    src.PhysicianNPI,
    src.PhysicianID,
    src.AdmitPhysicianName,
    src.AdmitPhysicianNPI,
    src.AdmitPhysicianID,
    src.ReferPhysicianName,
    src.ReferPhysicianNPI,
    src.ReferPhysicianID,
    src.PrimarySurgeonName,
    src.PrimarySurgeonNPI,
    src.PrimarySurgeonID,
    src.AdmitDate,
    src.DischargeDate,
    src.Registrar,
    src.ARStatus,
    src.PatType,
    src.Location,
    src.DRGCode,
    src.DRGDesc,
    src.DRGWeight,
    src.DRGVersion,
    src.AdmitFC,
    src.AdmitFinClass,
    src.CurrentFC,
    src.CurrentFinClass,
    src.TotalCharges,
    src.TotalPymt,
    src.TotalInsPymt,
    src.TotalPatientPymt,
    src.TotalAdj,
    src.TotalInsAdj,
    src.TotalPatientAdj,
    src.AcctBalance,
    src.InsBalance,
    src.PatBalance,
    src.ExpNetRev,
    src.PatFName,
    src.PatMName,
    src.PatLName,
    src.PatSuffix,
    src.PatDOB,
    src.PatSSN,
    src.PatGender,
    src.PatAddr1,
    src.PatAddr2,
    src.PatCity,
    src.PatState,
    src.PatZip,
    src.PatCountry,
    src.PatProvince,
    src.PatHomePhone,
    src.PatWorkPhone,
    src.PatCellPhone,
    src.PatEmployer,
    src.GuarRelationship,
    src.Guarantor,
    src.GuarFName,
    src.GuarMName,
    src.GuarLName,
    src.GuarSuffix,
    src.GuarDOB,
    src.GuarSSN,
    src.GuarGender,
    src.GuarAddr1,
    src.GuarAddr2,
    src.GuarCity,
    src.GuarState,
    src.GuarZip,
    src.GuarCountry,
    src.GuarProvince,
    src.GuarHomePhone,
    src.GuarWorkPhone,
    src.GuarCellPhone,
    src.GuarEmployer,
    src.LastBillDate,
    src.LastBillSubmitDate,
    src.LastBillType,
    src.LastBillMediaType,
    src.Agency,
    src.AgencyAssignDate,
    src.AgencyReturnDate,
    src.AgencyReturnReason,
    src.BadDebtDate,
    src.BadDebtAmt,
    src.BadDebtBal,
    src.ActiveCOB,
    src.PatientFacilityID,
    src.PatientFacilityName,
    src.PatientFacilityNPI,
    src.PatientFacilityType,
    src.CBSA,
    src.LocationCode,
    src.PatientStatusatBill,
    src.PatientStatusatCodeBill,
    src.LastClaimComment,
    src.StatementFromDate,
    src.StatementThroughDate,
    src.DateOfServiceFromDate,
    src.DateOfServiceThroughDate,
    src.MedicalDirectorName,
    src.MedicalDirectorNPI,
    src.MedicalDirectorID,
    src.BenefitPeriodID,
    src.BenefitPeriodStartDate,
    src.BenefitPeriodEndDate,
    src.BenefitPeriodStatus,
    src.EpisodeID,
    src.StartofEpisode,
    src.EndofEpisode,
    src.StartofPeriod,
    src.EndofPeriod,
    src.EpisodeStatus,
    src.ClaimFromDate,
    src.ClaimThroughDate,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)